In [1]:

# CELL 1 — PATH AND FILE VERIFICATION
import os
import sys

COMPONENT_PATH = r"C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals"
sys.path.insert(0, os.path.join(COMPONENT_PATH, "src"))

# Fixed import names matching your current config.py
from utils.config import (
    WESAD_PATH,
    WESAD_SUBJECT_IDS,
    SWELL_TRAIN,
    SWELL_TEST,
    SWELL_LABELS
)

print("=" * 60)
print("DATASET PATH VERIFICATION")
print("=" * 60)

# Check WESAD root
print(f"\nWESAD root: {WESAD_PATH}")
print(f"  Exists: {os.path.exists(WESAD_PATH)}")

# Check each WESAD subject
print("\nWESAD SUBJECTS:")
found = 0
for sid in WESAD_SUBJECT_IDS:
    pkl = os.path.join(WESAD_PATH, f"S{sid}", f"S{sid}.pkl")
    if os.path.exists(pkl):
        mb = os.path.getsize(pkl) / 1024 / 1024
        print(f"  [FOUND]   S{sid}  —  {mb:.0f} MB")
        found += 1
    else:
        print(f"  [MISSING] S{sid}  —  expected at: {pkl}")

print(f"\nTotal WESAD subjects found: {found} / 15")

# Check SWELL files
print("\nSWELL FILES:")
swell_files = {
    'Train  (CSV)':  SWELL_TRAIN,
    'Test   (CSV)':  SWELL_TEST,
    'Labels (XLSX)': SWELL_LABELS
}
for name, path in swell_files.items():
    exists = os.path.exists(path)
    status = "FOUND  " if exists else "MISSING"
    if exists:
        mb = os.path.getsize(path) / 1024 / 1024
        print(f"  [{status}]  {name}  ({mb:.1f} MB)")
    else:
        print(f"  [{status}]  {name}: {path}")

print("\nCell 1 complete")

Config loaded successfully
DATASET PATH VERIFICATION

WESAD root: C:\Users\weha1\datasets\WESAD
  Exists: True

WESAD SUBJECTS:
  [FOUND]   S2  —  930 MB
  [FOUND]   S3  —  984 MB
  [FOUND]   S4  —  986 MB
  [FOUND]   S5  —  947 MB
  [FOUND]   S6  —  1071 MB
  [FOUND]   S7  —  791 MB
  [FOUND]   S8  —  829 MB
  [FOUND]   S9  —  794 MB
  [FOUND]   S10  —  827 MB
  [FOUND]   S11  —  793 MB
  [FOUND]   S13  —  844 MB
  [FOUND]   S14  —  840 MB
  [FOUND]   S15  —  789 MB
  [FOUND]   S16  —  857 MB
  [FOUND]   S17  —  903 MB

Total WESAD subjects found: 15 / 15

SWELL FILES:
  [FOUND  ]  Train  (CSV)  (221.6 MB)
  [FOUND  ]  Test   (CSV)  (24.6 MB)
  [FOUND  ]  Labels (XLSX)  (0.2 MB)

Cell 1 complete


In [2]:

# CELL 2 — LOAD ONE WESAD SUBJECT AND VERIFY

import pickle
import numpy as np
import os
import sys

COMPONENT_PATH = r"C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals"
sys.path.insert(0, os.path.join(COMPONENT_PATH, "src"))

from utils.config import WESAD_PATH, WESAD_CHEST_FS

# Try loading S2 first
test_id   = 2
test_path = os.path.join(WESAD_PATH, f"S{test_id}", f"S{test_id}.pkl")

print(f"Loading S{test_id} from:")
print(f"  {test_path}")
print()

if not os.path.exists(test_path):
    print("FILE NOT FOUND")
    print("Check that WESAD subjects are directly at:")
    print(f"  {WESAD_PATH}\\S2\\S2.pkl")
    print("Not nested deeper like WESAD\\WESAD\\WESAD\\S2\\S2.pkl")
else:
    with open(test_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    chest  = data['signal']['chest']
    wrist  = data['signal']['wrist']
    labels = data['label'].flatten()

    print("=" * 55)
    print(f"SUBJECT S{test_id} LOADED SUCCESSFULLY")
    print("=" * 55)

    print(f"\nCHEST SIGNALS (from RespiBAN @ 700 Hz):")
    for sig_name in chest.keys():
        shape = chest[sig_name].shape
        print(f"  {sig_name:10s}  shape: {shape}")

    print(f"\nWRIST SIGNALS (from Empatica E4):")
    wrist_rates = {'BVP': 64, 'EDA': 4, 'ACC': 32, 'TEMP': 4}
    for sig_name in wrist.keys():
        shape = wrist[sig_name].shape
        rate  = wrist_rates.get(sig_name, '?')
        print(f"  {sig_name:10s}  shape: {shape}  @ {rate} Hz")

    print(f"\nLABEL INFORMATION:")
    label_names = {
        0: 'transient     → EXCLUDE',
        1: 'baseline      → Normal (0)',
        2: 'stress/TSST   → Stress (1)',
        3: 'amusement     → EXCLUDE',
        4: 'meditation    → EXCLUDE'
    }
    for lid in sorted(np.unique(labels)):
        count = int(np.sum(labels == lid))
        mins  = count / WESAD_CHEST_FS / 60
        name  = label_names.get(int(lid), 'unknown')
        print(f"  Label {int(lid)}  {name}")
        print(f"           {count} samples  =  {mins:.1f} minutes")

    # Quick sanity check on signal values
    print(f"\nSIGNAL VALUE SANITY CHECK:")
    ecg = chest['ECG'].flatten()
    eda = chest['EDA'].flatten()
    acc = chest['ACC']

    print(f"  ECG range:  {ecg.min():.4f} to {ecg.max():.4f} mV")
    print(f"  EDA range:  {eda.min():.4f} to {eda.max():.4f} uS")
    print(f"  ACC shape:  {acc.shape}  (N samples x 3 axes)")

    print(f"\nSubject ID in file: {data['subject']}")
    print(f"\nWESAD S{test_id} FULLY VERIFIED")

Loading S2 from:
  C:\Users\weha1\datasets\WESAD\S2\S2.pkl

SUBJECT S2 LOADED SUCCESSFULLY

CHEST SIGNALS (from RespiBAN @ 700 Hz):
  ACC         shape: (4255300, 3)
  ECG         shape: (4255300, 1)
  EMG         shape: (4255300, 1)
  EDA         shape: (4255300, 1)
  Temp        shape: (4255300, 1)
  Resp        shape: (4255300, 1)

WRIST SIGNALS (from Empatica E4):
  ACC         shape: (194528, 3)  @ 32 Hz
  BVP         shape: (389056, 1)  @ 64 Hz
  EDA         shape: (24316, 1)  @ 4 Hz
  TEMP        shape: (24316, 1)  @ 4 Hz

LABEL INFORMATION:
  Label 0  transient     → EXCLUDE
           2142701 samples  =  51.0 minutes
  Label 1  baseline      → Normal (0)
           800800 samples  =  19.1 minutes
  Label 2  stress/TSST   → Stress (1)
           430500 samples  =  10.2 minutes
  Label 3  amusement     → EXCLUDE
           253400 samples  =  6.0 minutes
  Label 4  meditation    → EXCLUDE
           537599 samples  =  12.8 minutes
  Label 6  unknown
           45500 samples  =  1

In [3]:

# CELL 3 — LOAD SWELL DATASET

import pandas as pd
import numpy as np
import os

# Correct paths — CSV not Excel for train/test
SWELL_TRAIN  = r"C:\Users\weha1\datasets\SWELL\archive\hrv dataset\data\final\train.csv"
SWELL_TEST   = r"C:\Users\weha1\datasets\SWELL\archive\hrv dataset\data\final\test.csv"
SWELL_LABELS = r"C:\Users\weha1\datasets\SWELL\archive\hrv dataset\data\raw\labels\hrv stress labels.xlsx"
SWELL_RRI    = r"C:\Users\weha1\datasets\SWELL\archive\hrv dataset\data\raw\rri"

print("=" * 65)
print("SWELL DATASET — FINAL LOAD")
print("=" * 65)

# ── Verify paths ────────────────────────────────────────────
print("\nPATH VERIFICATION:")
for name, path in {
    'Train  (CSV)': SWELL_TRAIN,
    'Test   (CSV)': SWELL_TEST,
    'Labels (XLSX)': SWELL_LABELS,
    'RRI folder':   SWELL_RRI
}.items():
    exists = os.path.exists(path)
    status = "FOUND  " if exists else "MISSING"
    if exists and os.path.isfile(path):
        mb = os.path.getsize(path) / 1024 / 1024
        print(f"  [{status}]  {name}  ({mb:.1f} MB)")
    else:
        print(f"  [{status}]  {name}")

print()

# ── Load train.csv ──────────────────────────────────────────
print("LOADING TRAIN FILE (CSV):")
try:
    train_df = pd.read_csv(SWELL_TRAIN)
    print(f"  Shape:   {train_df.shape}")
    print(f"  Columns ({len(train_df.columns)}):")
    for col in train_df.columns:
        print(f"    {col}")
    print(f"\n  First 3 rows:")
    print(train_df.head(3).to_string())
    train_ok = True
except Exception as e:
    print(f"  ERROR: {e}")
    train_ok = False

print()

# ── Load test.csv ───────────────────────────────────────────
print("LOADING TEST FILE (CSV):")
try:
    test_df = pd.read_csv(SWELL_TEST)
    print(f"  Shape:   {test_df.shape}")
    print(f"  Columns: {list(test_df.columns)}")
    test_ok = True
except Exception as e:
    print(f"  ERROR: {e}")
    test_ok = False

print()

# ── Load labels ─────────────────────────────────────────────
print("LOADING LABELS FILE (XLSX):")
try:
    labels_df = pd.read_excel(SWELL_LABELS)
    print(f"  Shape:   {labels_df.shape}")
    print(f"  Columns: {list(labels_df.columns)}")
    print(f"\n  First 3 rows:")
    print(labels_df.head(3).to_string())
    labels_ok = True
except Exception as e:
    print(f"  ERROR: {e}")
    labels_ok = False

print()

# ── Check RRI folder ────────────────────────────────────────
print("RRI FOLDER (raw RR intervals per subject):")
if os.path.exists(SWELL_RRI):
    rri_files = sorted(os.listdir(SWELL_RRI))
    print(f"  Found {len(rri_files)} subject files")
    for f in rri_files[:5]:
        kb = os.path.getsize(os.path.join(SWELL_RRI, f)) / 1024
        print(f"    {f}  ({kb:.0f} KB)")
    if len(rri_files) > 5:
        print(f"    ... and {len(rri_files)-5} more")

print()

# ── Map conditions to binary (Normal/Stress) ─────────────────
if labels_ok:
    print("=" * 65)
    print("CONDITION MAPPING TO BINARY (Normal/Stress)")
    print("=" * 65)

    condition_map = {
        'R': 0,   # Rest         → Normal (0)
        'N': 0,   # Neutral      → Normal (0)
        'T': 1,   # Time press   → Stress (1)
        'I': 1    # Interruption → Stress (1)
    }

    cond_col = None
    for col in ['Condition', 'condition', 'label', 'Label']:
        if col in labels_df.columns:
            cond_col = col
            break

    if cond_col:
        print(f"\n  Original values in '{cond_col}':")
        print(f"  {labels_df[cond_col].value_counts().to_dict()}")

        labels_df['pss_label'] = labels_df[cond_col].map(condition_map)

        print(f"\n  After mapping:")
        label_names = {0: 'Normal', 1: 'Stress'}
        for lid in sorted(labels_df['pss_label'].dropna().unique()):
            count = (labels_df['pss_label'] == lid).sum()
            print(f"    Class {int(lid)} "
                  f"({label_names[int(lid)]}): {count} rows")

        print(f"\n  Available HRV/EDA features in labels file:")
        feature_check = ['HR', 'RMSSD', 'SCL',
                         'MEAN_RR', 'SDNN', 'pNN50']
        for feat in feature_check:
            if feat in labels_df.columns:
                vals = labels_df[feat].dropna()
                print(f"    {feat}: "
                      f"mean={vals.mean():.3f}, "
                      f"min={vals.min():.3f}, "
                      f"max={vals.max():.3f}")

# ── Check train columns for label ───────────────────────────
if train_ok:
    print()
    print("=" * 65)
    print("CHECKING TRAIN FILE FOR LABEL COLUMN")
    print("=" * 65)
    found_label = False
    for col in ['label', 'Label', 'condition', 'Condition',
                'stress', 'class', 'target']:
        if col in train_df.columns:
            print(f"  Label column found: '{col}'")
            print(f"  Unique values: {train_df[col].unique()}")
            found_label = True
            break
    if not found_label:
        print("  No label column found in train")
        print("  Labels come from the separate labels file")

# ── Final summary ────────────────────────────────────────────
print()
print("=" * 65)
print("FINAL SUMMARY")
print("=" * 65)
print(f"  Train CSV:    {'LOADED' if train_ok  else 'FAILED'} "
      f"{'— ' + str(train_df.shape) if train_ok else ''}")
print(f"  Test CSV:     {'LOADED' if test_ok   else 'FAILED'} "
      f"{'— ' + str(test_df.shape) if test_ok else ''}")
print(f"  Labels XLSX:  {'LOADED' if labels_ok else 'FAILED'} "
      f"{'— ' + str(labels_df.shape) if labels_ok else ''}")

if train_ok and test_ok and labels_ok:
    print()
    print("ALL SWELL FILES LOADED SUCCESSFULLY")
    print("Ready to proceed to preprocessing")

print("\nCell 3 complete")

SWELL DATASET — FINAL LOAD

PATH VERIFICATION:
  [FOUND  ]  Train  (CSV)  (221.6 MB)
  [FOUND  ]  Test   (CSV)  (24.6 MB)
  [FOUND  ]  Labels (XLSX)  (0.2 MB)
  [FOUND  ]  RRI folder

LOADING TRAIN FILE (CSV):
  Shape:   (369289, 36)
  Columns (36):
    MEAN_RR
    MEDIAN_RR
    SDRR
    RMSSD
    SDSD
    SDRR_RMSSD
    HR
    pNN25
    pNN50
    SD1
    SD2
    KURT
    SKEW
    MEAN_REL_RR
    MEDIAN_REL_RR
    SDRR_REL_RR
    RMSSD_REL_RR
    SDSD_REL_RR
    SDRR_RMSSD_REL_RR
    KURT_REL_RR
    SKEW_REL_RR
    VLF
    VLF_PCT
    LF
    LF_PCT
    LF_NU
    HF
    HF_PCT
    HF_NU
    TP
    LF_HF
    HF_LF
    sampen
    higuci
    datasetId
    condition

  First 3 rows:
      MEAN_RR   MEDIAN_RR        SDRR      RMSSD       SDSD  SDRR_RMSSD         HR      pNN25     pNN50        SD1         SD2      KURT      SKEW  MEAN_REL_RR  MEDIAN_REL_RR  SDRR_REL_RR  RMSSD_REL_RR  SDSD_REL_RR  SDRR_RMSSD_REL_RR  KURT_REL_RR  SKEW_REL_RR          VLF    VLF_PCT           LF     LF_PCT      

In [4]:

# CELL 4 — LOAD ALL WESAD SUBJECTS SUMMARY
import pickle
import numpy as np
import os
import sys

COMPONENT_PATH = r"C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals"
sys.path.insert(0, os.path.join(COMPONENT_PATH, "src"))

from utils.config import WESAD_PATH, WESAD_SUBJECT_IDS, WESAD_CHEST_FS

print("=" * 65)
print("LOADING ALL WESAD SUBJECTS")
print("=" * 65)

summary = []
failed  = []

label_names = {
    1: 'Baseline',
    2: 'Stress/TSST',
    3: 'Amusement',
    4: 'Meditation'
}

for sid in WESAD_SUBJECT_IDS:
    pkl_path = os.path.join(WESAD_PATH, f"S{sid}", f"S{sid}.pkl")

    if not os.path.exists(pkl_path):
        print(f"  [MISSING] S{sid}")
        failed.append(sid)
        continue

    with open(pkl_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    labels = data['label'].flatten()
    chest  = data['signal']['chest']

    # Count minutes per condition
    baseline_min = int(np.sum(labels == 1)) / WESAD_CHEST_FS / 60
    stress_min   = int(np.sum(labels == 2)) / WESAD_CHEST_FS / 60
    total_min    = len(labels) / WESAD_CHEST_FS / 60

    print(f"  [LOADED]  S{sid:2d}  |  "
          f"Total: {total_min:.0f}min  |  "
          f"Baseline: {baseline_min:.0f}min  |  "
          f"Stress: {stress_min:.0f}min")

    summary.append({
        'subject': sid,
        'total_min': total_min,
        'baseline_min': baseline_min,
        'stress_min': stress_min,
        'ecg_shape': chest['ECG'].shape,
        'eda_shape': chest['EDA'].shape
    })

print()
print("=" * 65)
print(f"SUCCESSFULLY LOADED: {len(summary)} / 15 subjects")
if failed:
    print(f"FAILED: {failed}")

print()
print("AGGREGATE STATISTICS:")
if summary:
    avg_baseline = np.mean([s['baseline_min'] for s in summary])
    avg_stress   = np.mean([s['stress_min']   for s in summary])
    avg_total    = np.mean([s['total_min']    for s in summary])
    print(f"  Average total recording:  {avg_total:.1f} minutes")
    print(f"  Average baseline (Normal): {avg_baseline:.1f} minutes")
    print(f"  Average stress (TSST):     {avg_stress:.1f} minutes")
    print(f"  Estimated total windows (60s, 50% overlap):")
    total_windows_est = len(summary) * (avg_baseline + avg_stress) * 60 / 30
    print(f"    ~{int(total_windows_est)} windows across all subjects")

print()
print("ALL WESAD SUBJECTS LOADED SUCCESSFULLY")
print("Ready to proceed to preprocessing")

LOADING ALL WESAD SUBJECTS
  [LOADED]  S 2  |  Total: 101min  |  Baseline: 19min  |  Stress: 10min
  [LOADED]  S 3  |  Total: 108min  |  Baseline: 19min  |  Stress: 11min
  [LOADED]  S 4  |  Total: 107min  |  Baseline: 19min  |  Stress: 11min
  [LOADED]  S 5  |  Total: 104min  |  Baseline: 20min  |  Stress: 11min
  [LOADED]  S 6  |  Total: 118min  |  Baseline: 20min  |  Stress: 11min
  [LOADED]  S 7  |  Total: 87min  |  Baseline: 20min  |  Stress: 11min
  [LOADED]  S 8  |  Total: 91min  |  Baseline: 19min  |  Stress: 11min
  [LOADED]  S 9  |  Total: 87min  |  Baseline: 20min  |  Stress: 11min
  [LOADED]  S10  |  Total: 92min  |  Baseline: 20min  |  Stress: 12min
  [LOADED]  S11  |  Total: 87min  |  Baseline: 20min  |  Stress: 11min
  [LOADED]  S13  |  Total: 92min  |  Baseline: 20min  |  Stress: 11min
  [LOADED]  S14  |  Total: 92min  |  Baseline: 20min  |  Stress: 11min
  [LOADED]  S15  |  Total: 88min  |  Baseline: 20min  |  Stress: 11min
  [LOADED]  S16  |  Total: 94min  |  Baseline